# Track 1: ЭЭГ → изображение в две стадии

Задача: по окну ЭЭГ (−0.2…0.8 с вокруг показа картинки) предсказать эмбеддинг картинки DINOv2-giant (1536 чисел).
Метрика — top-5: предсказание сравнивается по косинусу со всеми test-картинками, которые модель не видела при обучении.

1. **CLIP-энкодер** (`encoder.py`). ЭЭГ → $e$ в центрированном пространстве целей $z=(y-\mu)/s$:
   $L=\mathrm{CE}\big(\tau\cos(e,z)^\top, P\big)$, где $P_{ij}\propto[\text{картинка}_i=\text{картинка}_j]$, $\tau$ обучается.
   **Канал пациента:** у каждого человека обучаемый временной ряд длины $T$, он добавляется к ЭЭГ как ещё один канал.
   Новому человеку достаётся среднее выученных каналов; при обучении свой канал иногда подменяется средним.
   Платформа передаёт только сигнал, поэтому там всегда средний канал.
2. **Flow-matching приор** (`prior.py`). Уточняет $e$: выпрямленный поток от предсказания к настоящему эмбеддингу,
   $x_0=\hat e+\sigma\varepsilon$, $x_1=z$, $x_t=(1-t)x_0+t\,x_1$, $L=\|v(x_t,t,\hat e)-(x_1-x_0)\|^2$.
   $v$ — U-Net для векторов (1536→1024→512→1024→1536 со skip-связями, время и условие через FiLM).
   Учится на val-окнах: энкодер их не видел, поэтому пары $(\hat e, z)$ честные.

Отправка одной командой из папки `track1_image`: `python -m image_decoding.make_submission --study xu2025alljoined`.


In [ ]:
import sys; sys.path.append("..")
import pandas as pd, torch
from image_decoding import data
from image_decoding.encoder import train_encoder
from image_decoding.prior import embed_split, fit_prior, refined_prediction, split_by_image
from image_decoding.evaluation import evaluate, retrieval, REFERENCE

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SYNTHETIC = False  # True: маленькие фейковые данные, чтобы проверить код без скачивания
if SYNTHETIC:
    loaders, meta = data.synthetic()
else:
    loaders, meta = data.loaders("xu2025alljoined", "../data", batch_size=256)
EPOCHS = (8, 60) if SYNTHETIC else (30, 200)  # (энкодер, приор)
print({k: v for k, v in meta.items() if k != "ch_names"}, {name: len(l.dataset) for name, l in loaders.items()})

## Стадия 1: CLIP-энкодер
Обучение на train, лучшая эпоха — по top-5 на val (галерея — картинки val, человек считается неизвестным).

In [ ]:
model, history = train_encoder(loaders, meta, epochs=EPOCHS[0], device=DEVICE)
pd.DataFrame(history).set_index("epoch")[["top5", "top1"]].plot(title="val, %");

## Стадия 2: flow-matching приор
Пары (выход энкодера, настоящий эмбеддинг) на val-окнах. 10% val-картинок отложены, чтобы выбрать эпоху и число шагов.

In [ ]:
e, y, s = embed_split(model, loaders["val"])
held = split_by_image(y, fraction=0.1)
history = fit_prior(model, e[~held], y[~held], epochs=EPOCHS[1], monitor=(e[held], y[held], s[held]))
pd.DataFrame(history).set_index("epoch")[["top5"]].plot(title="отложенные val-картинки, top-5 %");

Сколько шагов Эйлера нужно. 0 шагов — это просто выход энкодера, то есть стадия 1.

In [ ]:
steps = {n: retrieval(refined_prediction(model, e[held], n), y[held], s[held])["top5"]
         for n in [0, 1, 2, 5, 10, 20]}
model.n_steps = max(steps, key=steps.get)
pd.Series(steps).plot(marker="o", xlabel="шаги", title="top-5 %, отложенные val-картинки")
print("n_steps =", model.n_steps)

## Test: новые картинки
`top5` — каждое окно отдельно (warm-up), `top5_subject_agg` — повторы картинки усреднены внутри человека (закрытый тест).
«Средний канал» — как на платформе (человек неизвестен), «свой канал» — человек из train.

In [ ]:
results = {}
for name, n_steps in [("стадия 1", 0), ("стадия 1 + приор", model.n_steps)]:
    keep, model.n_steps = model.n_steps, n_steps
    results[name + ", средний канал"] = evaluate(model, loaders["test"])
    results[name + ", свой канал"] = evaluate(model, loaders["test"], known_subject=True)
    model.n_steps = keep
print("ориентиры на THINGS-EEG2, top-5 %:", REFERENCE)
pd.DataFrame(results).T.round(2)